# Day 4: NumPy and Pandas for AI Data Engineering

## From Fast Arrays to Machine Learning-Ready DataFrames

This notebook moves from low-level numerical arrays into high-level DataFrame engineering.

Today students will learn:

- Why NumPy is faster than normal Python lists
- How broadcasting works
- How boolean masks filter data
- How image-like data is represented as arrays
- How Pandas DataFrames organize tabular data
- How to inspect, clean, group, merge, and save datasets
- How to convert cleaned DataFrames into NumPy arrays for ML models

This is an important bridge between Python basics and real machine learning pipelines.

---
# 1. NumPy Speed and Memory

NumPy operations occur in optimized compiled code. We avoid Python loops whenever possible when scaling.

In AI, this matters because datasets can contain millions of pixels, embeddings, or sensor readings.

This example compares a Python list comprehension with NumPy vectorization.

In [1]:
import numpy as np
import time

data_size = 10_000_000

large_list = list(range(data_size))
large_array = np.arange(data_size)

start = time.time()
squared_list = [x**2 for x in large_list]
print(f"Python List Comprehension: {time.time() - start:.4f} seconds")

start = time.time()
squared_array = large_array ** 2
print(f"NumPy Vectorization: {time.time() - start:.4f} seconds")

Python List Comprehension: 1.1453 seconds
NumPy Vectorization: 0.0049 seconds


## Teaching Note

Your original example used 10 million values. That is good for a powerful system, but 1 million is safer for student laptops and Colab.

You can change `data_size` to `10_000_000` during live demonstration if the machine can handle it.

---
# 2. NumPy Array Shape

In ML, shape is extremely important.

Examples:

| Data | Shape |
|---|---|
| One grayscale image | `(height, width)` |
| One RGB image | `(height, width, 3)` |
| Batch of images | `(batch, height, width, channels)` |
| Student table with 4 rows and 2 scores | `(4, 2)` |

In [2]:
student_matrix = np.array([
    [85, 90],
    [78, 88],
    [92, 95],
    [60, 72]
])

print("Student Score Matrix:")
print(student_matrix)

print("Shape:", student_matrix.shape)
print("Dimensions:", student_matrix.ndim)
print("Total values:", student_matrix.size)

Student Score Matrix:
[[85 90]
 [78 88]
 [92 95]
 [60 72]]
Shape: (4, 2)
Dimensions: 2
Total values: 8


---
# 3. Broadcasting and Array Math

Broadcasting allows us to perform mathematical operations on arrays without manually looping.

Here, a bonus of 50 is applied to every computer vision score instantly.

In [3]:
cv_scores = np.array([85, 92, 45, 90, 78])

adjusted_scores = cv_scores + 50
print("Original Scores:", cv_scores)
print("Scores after bonus broadcast:", adjusted_scores)

high_achiever_mask = adjusted_scores >= 140
print("Boolean Mask:", high_achiever_mask)
print("Extracted Top Scores:", adjusted_scores[high_achiever_mask])

Original Scores: [85 92 45 90 78]
Scores after bonus broadcast: [135 142  95 140 128]
Boolean Mask: [False  True False  True False]
Extracted Top Scores: [142 140]


## Visualization: Before and After Broadcasting

In [ ]:
import matplotlib.pyplot as plt

student_labels = [f"S{i}" for i in range(1, len(cv_scores) + 1)]

plt.figure(figsize=(8, 5))
plt.plot(student_labels, cv_scores, marker="o", label="Original Scores")
plt.plot(student_labels, adjusted_scores, marker="o", label="Adjusted Scores")
plt.title("Broadcasting: Adding Bonus to All Scores")
plt.xlabel("Student")
plt.ylabel("Score")
plt.legend()
plt.show()

---
# 4. Boolean Masking

Boolean masking filters arrays using conditions.

This is useful for selecting high-confidence predictions, removing invalid values, or isolating top-performing samples.

In [ ]:
prediction_confidence = np.array([0.91, 0.52, 0.88, 0.34, 0.97, 0.76])

accepted_mask = prediction_confidence >= 0.80
accepted_predictions = prediction_confidence[accepted_mask]

print("All Predictions:", prediction_confidence)
print("Accepted Mask:", accepted_mask)
print("Accepted Predictions:", accepted_predictions)

---
# 5. Image-Like Array Example

Images are arrays.

A grayscale image is a 2D matrix where each number represents pixel intensity.

Usually:

- `0` means black
- `255` means white

Normalizing pixel values by dividing by 255 is a common preprocessing step before training neural networks.

In [ ]:
image_patch = np.array([
    [0, 50, 100, 150, 200],
    [25, 75, 125, 175, 225],
    [50, 100, 150, 200, 250],
    [75, 125, 175, 225, 255],
    [100, 150, 200, 250, 255]
])

normalized_patch = image_patch / 255.0

print("Original Image Patch:")
print(image_patch)

print("\nNormalized Image Patch:")
print(np.round(normalized_patch, 3))

In [ ]:
plt.figure(figsize=(5, 5))
plt.imshow(image_patch, cmap="gray")
plt.title("5x5 Grayscale Image Patch")
plt.colorbar(label="Pixel Intensity")
plt.show()

---
# 6. Pandas: Creating the DataFrame

DataFrames organize 1D arrays into labeled 2D tables.

A DataFrame is like an Excel sheet inside Python.

Here we simulate a grading dataset for 123 students.

In [ ]:
import pandas as pd

np.random.seed(42)

student_ids = [f"S-{i:03}" for i in range(1, 124)]
vision_scores = np.random.randint(40, 100, size=123)
text_scores = np.random.randint(50, 100, size=123)

df_grades = pd.DataFrame({
    "Student_ID": student_ids,
    "Vision_Track": vision_scores,
    "Text_Track": text_scores,
})

print("Autograder Dataset Initialized:")
display(df_grades.head())

Autograder Dataset Initialized:


,Student_ID,Vision_Track,Text_Track,Text_Track2
0,S-001,78,74,74
1,S-002,91,56,56
2,S-003,68,58,58
3,S-004,54,73,73
4,S-005,82,50,50


---
# 7. DataFrame Inspection

Before modeling, always inspect the data.

Useful commands:

- `.head()`
- `.tail()`
- `.shape`
- `.columns`
- `.info()`
- `.describe()`

In [7]:
print("Shape:", df_grades.shape)

print("\nColumns:")
print(df_grades.columns)

print("\nInfo:")
df_grades.info()

print("\nStatistics:")
display(df_grades.describe())

Shape: (123, 4)

Columns:
Index(['Student_ID', 'Vision_Track', 'Text_Track', 'Text_Track2'], dtype='object')

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123 entries, 0 to 122
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Student_ID    123 non-null    object
 1   Vision_Track  123 non-null    int64 
 2   Text_Track    123 non-null    int64 
 3   Text_Track2   123 non-null    int64 
dtypes: int64(3), object(1)
memory usage: 4.0+ KB

Statistics:


,Vision_Track,Text_Track,Text_Track2
count,123.000000,123.000000,123.000000
mean,70.569106,74.756098,74.756098
std,18.069121,14.607623,14.607623
min,40.000000,50.000000,50.000000
25%,55.000000,61.500000,61.500000
50%,68.000000,77.000000,77.000000
75%,86.000000,87.000000,87.000000
max,99.000000,98.000000,98.000000


---
# 8. DataFrame Math and Feature Engineering

Pandas allows column-to-column math.

Feature engineering means creating new useful columns from existing columns.

Here we create `Combined_Score`.

In [ ]:
df_grades["Combined_Score"] = (df_grades["Vision_Track"] + df_grades["Text_Track"]) / 2

df_top = df_grades.sort_values(by="Combined_Score", ascending=False)

print("Top 3 Students:")
display(df_top.head(3))

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df_grades["Combined_Score"], bins=15, edgecolor="black")
plt.title("Distribution of Combined Scores")
plt.xlabel("Combined Score")
plt.ylabel("Number of Students")
plt.show()

---
# 9. Filtering Rows in Pandas

Filtering in Pandas is similar to boolean masking in NumPy.

Here we select students with combined score greater than or equal to 85.

In [ ]:
strong_students = df_grades[df_grades["Combined_Score"] >= 85]

print(f"Number of strong students: {len(strong_students)}")
display(strong_students.head())

---
# 10. Data Cleaning: Handling Nulls

Neural networks cannot process missing values directly.

Missing values are often represented as:

- `NaN`
- `None`

Here we drop rows with missing critical labels and fill missing numerical values with the column mean.

In [ ]:
df_messy = pd.DataFrame({
    "Doc_ID": ["D1", "D2", "D3", "D4"],
    "Morphology_Score": [0.88, np.nan, 0.91, 0.74],
    "Language_Tag": ["Dravidian", "Dravidian", None, "Indo-Aryan"]
})

print("--- Raw Messy Dataset ---")
display(df_messy)

df_clean = df_messy.dropna(subset=["Language_Tag"]).copy()

mean_score = df_clean["Morphology_Score"].mean()
df_clean["Morphology_Score"] = df_clean["Morphology_Score"].fillna(mean_score)

print("\n--- Cleaned Dataset Ready for Modeling ---")
display(df_clean)

---
# 11. Detecting Missing Values

Before cleaning data, check where values are missing.

`.isna().sum()` gives the number of missing values per column.

In [ ]:
print("Missing value count per column:")
display(df_messy.isna().sum())

print("\nMissing value mask:")
display(df_messy.isna())

---
# 12. The `apply()` Method and Grouping

`apply()` lets us run a custom function across a column.

`groupby()` follows:

```text
Split → Apply → Combine
```

Here we classify students into performance tiers and then calculate average module scores per tier.

In [ ]:
def assign_tier(score):
    if score >= 140:
        return "Distinction"
    elif score >= 100:
        return "Pass"
    else:
        return "Review"

df_grades["Status"] = (df_grades["Combined_Score"] + 50).apply(assign_tier)

tier_metrics = df_grades.groupby("Status")[["Vision_Track", "Text_Track"]].mean()

print("Average Module Performance by Tier:")
display(tier_metrics)

In [ ]:
tier_metrics.plot(kind="bar", figsize=(8, 5))
plt.title("Average Module Performance by Tier")
plt.xlabel("Status")
plt.ylabel("Average Score")
plt.xticks(rotation=0)
plt.show()

---
# 13. Category Counts

`value_counts()` tells us how many records belong to each category.

This is useful for checking label balance in ML datasets.

In [ ]:
print("Number of students in each status:")
display(df_grades["Status"].value_counts())

---
# 14. Merging DataFrames

Real projects often use multiple files.

For example:

- One file contains scores
- One file contains departments
- One file contains attendance

`merge()` combines DataFrames using a shared key, similar to SQL joins.

In [ ]:
df_departments = pd.DataFrame({
    "Student_ID": [f"S-{i:03}" for i in range(1, 124)],
    "Department": np.random.choice(["EEE", "CSE", "ECE", "MECH"], size=123)
})

df_full = pd.merge(df_grades, df_departments, on="Student_ID", how="left")

print("Merged Dataset:")
display(df_full.head())

---
# 15. Department-Level Analysis

After merging, we can group by department and compare average performance.

In [ ]:
department_summary = df_full.groupby("Department")[["Vision_Track", "Text_Track", "Combined_Score"]].mean()

print("Average Scores by Department:")
display(department_summary)

In [ ]:
department_summary["Combined_Score"].plot(kind="bar", figsize=(8, 5))
plt.title("Average Combined Score by Department")
plt.xlabel("Department")
plt.ylabel("Average Combined Score")
plt.xticks(rotation=0)
plt.show()

---
# 16. Saving Cleaned Data

After cleaning and feature engineering, save the processed dataset.

CSV is simple and easy to share.

In [ ]:
output_file = "cleaned_autograder_dataset.csv"

df_full.to_csv(output_file, index=False)

print(f"Cleaned dataset saved as: {output_file}")

---
# 17. Reading Saved Data Back

This simulates a common ML workflow:

1. Clean raw data
2. Save processed data
3. Load processed data later for modeling

In [ ]:
df_loaded = pd.read_csv("cleaned_autograder_dataset.csv")

print("Loaded saved dataset:")
display(df_loaded.head())

---
# 18. ML-Ready Dataset Check

Before giving data to a model, check:

- Shape
- Missing values
- Numeric feature columns
- Data types

In [ ]:
print("Dataset Shape:", df_loaded.shape)

print("\nMissing Values:")
display(df_loaded.isna().sum())

feature_columns = ["Vision_Track", "Text_Track", "Combined_Score"]

print("\nSelected ML Feature Columns:")
display(df_loaded[feature_columns].head())

print("\nFeature Data Types:")
display(df_loaded[feature_columns].dtypes)

---
# 19. Converting DataFrame Columns to NumPy Arrays

Pandas is excellent for cleaning.

NumPy is excellent for numerical computation.

Many ML libraries expect NumPy arrays, so we often convert DataFrame columns into arrays.

In [ ]:
X = df_loaded[["Vision_Track", "Text_Track"]].to_numpy()
y = df_loaded["Combined_Score"].to_numpy()

print("Feature Matrix X shape:", X.shape)
print("Target Array y shape:", y.shape)

print("\nFirst 5 rows of X:")
print(X[:5])

print("\nFirst 5 values of y:")
print(y[:5])

---
# Day 4 Hands-On Coding Test

The following problems are for students to solve independently.

No solutions are provided in this notebook section.

# Test 1: Easy  
## NumPy Broadcasting and Masking

You are given quiz scores:

```python
scores = np.array([45, 78, 88, 92, 56, 99])
```

Write Python code to:

1. Add a bonus of `10` marks to every score using broadcasting
2. Print the adjusted scores
3. Create a boolean mask for scores greater than or equal to `90`
4. Print only the high scores

In [ ]:
# Test 1 Student Code

import numpy as np

scores = np.array([45, 78, 88, 92, 56, 99])

# Write your solution here

---
# Test 2: Medium  
## Pandas Cleaning and Feature Engineering

You are given a messy student dataset:

```python
data = {
    "Student_ID": ["S1", "S2", "S3", "S4"],
    "Math_Score": [80, np.nan, 90, 70],
    "AI_Score": [85, 88, np.nan, 75]
}
```

Write Python code to:

1. Create a DataFrame from the dictionary
2. Fill missing values in `Math_Score` with the mean of `Math_Score`
3. Fill missing values in `AI_Score` with the mean of `AI_Score`
4. Create a new column called `Average_Score`
5. Display the final cleaned DataFrame

In [ ]:
# Test 2 Student Code

import numpy as np
import pandas as pd

data = {
    "Student_ID": ["S1", "S2", "S3", "S4"],
    "Math_Score": [80, np.nan, 90, 70],
    "AI_Score": [85, 88, np.nan, 75]
}

# Write your solution here

---
# Test 3: Hard  
## GroupBy and Department-Level Analysis

You are given:

```python
students = pd.DataFrame({
    "Student_ID": ["S1", "S2", "S3", "S4", "S5", "S6"],
    "Department": ["EEE", "CSE", "EEE", "ECE", "CSE", "ECE"],
    "Vision_Score": [82, 91, 76, 88, 95, 69],
    "Text_Score": [78, 89, 80, 92, 90, 72]
})
```

Write Python code to:

1. Create `Combined_Score`
2. Create `assign_status(score)`
3. Return `"Excellent"` if score is `>= 90`
4. Return `"Good"` if score is `>= 75`
5. Return `"Review"` otherwise
6. Use `.apply()` to create a `Status` column
7. Use `groupby()` to calculate average combined score for each department
8. Create a bar chart showing average combined score by department

In [ ]:
# Test 3 Student Code

import pandas as pd
import matplotlib.pyplot as plt

students = pd.DataFrame({
    "Student_ID": ["S1", "S2", "S3", "S4", "S5", "S6"],
    "Department": ["EEE", "CSE", "EEE", "ECE", "CSE", "ECE"],
    "Vision_Score": [82, 91, 76, 88, 95, 69],
    "Text_Score": [78, 89, 80, 92, 90, 72]
})

# Write your solution here

---
# End of Day 4 Notebook

By the end of this notebook, students should understand:

- Why NumPy is faster than regular Python loops
- How broadcasting works
- How boolean masking filters arrays
- How image-like data can be represented as arrays
- How to create and inspect Pandas DataFrames
- How to engineer new features
- How to filter rows
- How to handle missing values
- How to use `apply()` and `groupby()`
- How to merge DataFrames
- How to save and reload cleaned datasets
- How to convert DataFrames into NumPy arrays for ML models